In [13]:
import os
import time
import pandas as pd
import numpy as np
import joblib
from sklearn import tree
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import StandardScaler
# 如果你的 data_utils.py 和这个notebook在同一个目录下或Python路径可找到
# from data_utils import CustomDataset, load_presplit_data_from_dir # 虽然DT用不到，但保持完整性
def load_and_preprocess_data(presplit_dir, label_col, train_file, test_file, standardize_all_features=True, num_cols_to_standardize=None):
    """
    从预分割的CSV加载数据，并进行标准化。
    standardize_all_features=True: 对所有特征标准化。
    standardize_all_features=False and num_cols_to_standardize is not None: 只对前num_cols_to_standardize列标准化。
    """
    train_file_path = os.path.join(presplit_dir, train_file)
    test_file_path = os.path.join(presplit_dir, test_file)

    if not os.path.exists(train_file_path):
        raise FileNotFoundError(f"预分割的训练数据文件未找到: {train_file_path}")
    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"预分割的测试数据文件未找到: {test_file_path}")

    print(f"从预分割文件加载数据:")
    print(f"  训练数据: {train_file_path}")
    print(f"  测试数据: {test_file_path}")

    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)

    if label_col not in train_df.columns or label_col not in test_df.columns:
        raise ValueError(f"标签列 '{label_col}' 未在预分割的CSV文件中找到。")

    X_train_raw = train_df.drop(label_col, axis=1)
    y_train = train_df[label_col].to_numpy()
    X_test_raw = test_df.drop(label_col, axis=1)
    y_test = test_df[label_col].to_numpy()

    feature_names = X_train_raw.columns.tolist()
    
    # --- 特征标准化 ---
    if standardize_all_features:
        print("对所有特征列进行标准化 (基于训练集拟合)...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)
    elif num_cols_to_standardize is not None and num_cols_to_standardize > 0:
        print(f"将对前 {num_cols_to_standardize} 列特征进行标准化 (基于训练集拟合)...")
        X_train_to_scale = X_train_raw.iloc[:, :num_cols_to_standardize]
        X_train_not_scaled = X_train_raw.iloc[:, num_cols_to_standardize:]

        X_test_to_scale = X_test_raw.iloc[:, :num_cols_to_standardize]
        X_test_not_scaled = X_test_raw.iloc[:, num_cols_to_standardize:]

        scaler = StandardScaler()
        X_train_scaled_part = scaler.fit_transform(X_train_to_scale)
        X_test_scaled_part = scaler.transform(X_test_to_scale)
        
        # 确保索引对齐以便拼接（如果原始是Pandas DataFrame则更简单）
        # 如果X_train_not_scaled 和 X_test_not_scaled 是numpy array，则用np.hstack
        if isinstance(X_train_not_scaled, pd.DataFrame):
             X_train_scaled = pd.concat([pd.DataFrame(X_train_scaled_part, columns=X_train_to_scale.columns, index=X_train_to_scale.index),
                                      X_train_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
             X_test_scaled = pd.concat([pd.DataFrame(X_test_scaled_part, columns=X_test_to_scale.columns, index=X_test_to_scale.index),
                                     X_test_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
        else: # 假设是numpy array
            X_train_scaled = np.hstack((X_train_scaled_part, X_train_not_scaled.to_numpy() if hasattr(X_train_not_scaled, 'to_numpy') else X_train_not_scaled))
            X_test_scaled = np.hstack((X_test_scaled_part, X_test_not_scaled.to_numpy() if hasattr(X_test_not_scaled, 'to_numpy') else X_test_not_scaled))
    else:
        print("不进行特征标准化处理。")
        X_train_scaled = X_train_raw.to_numpy()
        X_test_scaled = X_test_raw.to_numpy()


    print(f"数据加载和预处理完成。训练特征形状: {X_train_scaled.shape}, 测试特征形状: {X_test_scaled.shape}")
    return X_train_scaled, y_train, X_test_scaled, y_test, feature_names
# --- 手动设置参数 (模拟命令行参数) ---
class Args: # 创建一个简单的类来模拟 argparse 的 Namespace 对象
    pass

args = Args()

# === 配置你要测试的数据集和模型参数 ===
# 1. 指定预分割数据所在的目录
args.presplit_data_dir = './temp_data_utils_logs/mimic3_36_factors_split' # 例如 MIMIC-III 36因子
# args.presplit_data_dir = './temp_data_utils_logs/local_8_factors_split'   # 或者 Local 8因子

# 2. 指定输出目录 (确保这个目录存在或脚本可以创建它)
#    为每个实验创建一个唯一的输出目录
data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir)) # e.g., mimic3_36_factors_split
args.output_dir = f'./traditional_ml_outputs/DT_{data_source_tag.replace("_split", "")}_notebook'

# 3. 标签列名
args.label_column = "dead"

# 4. 决策树参数 (与原始代码一致)
args.criterion = 'entropy'
args.splitter = 'random'

# 5. 标准化选项 (根据你的原始 stander_data 行为调整)
# 选项 A: 对所有特征进行标准化 (更标准的做法)
standardize_all_features_flag = True
num_cols_to_standardize_value = None # 仅当 standardize_all_features_flag = False 时使用

# 选项 B: 只对前N列特征进行标准化 (更接近你原始stander_data的行为，但需要知道N)
# standardize_all_features_flag = False
# num_cols_to_standardize_value = 8 # 或 18，根据你的 stander_data 逻辑

# 选项 C: 完全不进行标准化 (如果你的原始数据已经是某种形式的"标准化"或你不想标准化)
# standardize_all_features_flag = False
# num_cols_to_standardize_value = None
# =======================================


# --- 开始执行训练和评估逻辑 ---
os.makedirs(args.output_dir, exist_ok=True)
print(f"所有输出将保存到: {args.output_dir}")
print(f"使用的参数: {vars(args)}")


# 1. 加载和预处理数据
X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
    presplit_dir=args.presplit_data_dir,
    label_col=args.label_column,
    train_file="split_train_data_seed256.csv",
    test_file="split_test_data_seed256.csv",
    standardize_all_features=standardize_all_features_flag,
    num_cols_to_standardize=num_cols_to_standardize_value
)

# 2. 训练决策树模型
print(f"\n开始训练决策树模型: criterion='{args.criterion}', splitter='{args.splitter}'")
start_time = time.time()

base_dt_model = tree.DecisionTreeClassifier(
    criterion=args.criterion,
    splitter=args.splitter,
    random_state=256 # 与数据分割种子一致，确保决策树训练本身也可复现
)

# 使用 CalibratedClassifierCV 进行概率校准
calibrated_model = CalibratedClassifierCV(base_dt_model, method='isotonic', cv=3) # 也可以尝试 method='sigmoid'
calibrated_model.fit(X_train, y_train)

model_save_path = os.path.join(args.output_dir, "decision_tree_calibrated.joblib")
joblib.dump(calibrated_model, model_save_path)
print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

# 3. 在测试集上评估并获取所需输出
y_pred_probs_test = calibrated_model.predict_proba(X_test)
y_pred_class_test = calibrated_model.predict(X_test)
positive_class_probs_test = y_pred_probs_test[:, 1]

test_acc = accuracy_score(y_test, y_pred_class_test)
test_auc = roc_auc_score(y_test, positive_class_probs_test)
test_f1 = f1_score(y_test, y_pred_class_test, average='binary')
test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
test_cm = confusion_matrix(y_test, y_pred_class_test)

print("\n--- 测试集评估结果 ---")
print(f"  准确率 (Accuracy): {test_acc:.4f}")
print(f"  AUC: {test_auc:.4f}")
print(f"  F1 分数: {test_f1:.4f}")
print(f"  MCC: {test_mcc:.4f}")
print(f"  混淆矩阵:\n{test_cm}")

# 4. 保存DeLong检验所需的文件
np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

# 5. 保存指标摘要
metrics_summary = {
    "model_name": "DecisionTree",
    "presplit_data_dir": args.presplit_data_dir,
    "criterion": args.criterion,
    "splitter": args.splitter,
    "test_accuracy": test_acc,
    "test_auc": test_auc,
    "test_f1": test_f1,
    "test_mcc": test_mcc,
    "confusion_matrix_tn": test_cm[0,0],
    "confusion_matrix_fp": test_cm[0,1],
    "confusion_matrix_fn": test_cm[1,0],
    "confusion_matrix_tp": test_cm[1,1],
}
metrics_df = pd.DataFrame([metrics_summary])
metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

print("\n决策树模型训练和评估完成。")

所有输出将保存到: ./traditional_ml_outputs/DT_mimic3_36_factors_notebook
使用的参数: {'presplit_data_dir': './temp_data_utils_logs/mimic3_36_factors_split', 'output_dir': './traditional_ml_outputs/DT_mimic3_36_factors_notebook', 'label_column': 'dead', 'criterion': 'entropy', 'splitter': 'random'}
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (2667, 36), 测试特征形状: (1144, 36)

开始训练决策树模型: criterion='entropy', splitter='random'
模型训练完成并在 0.054s 内保存至 ./traditional_ml_outputs/DT_mimic3_36_factors_notebook/decision_tree_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.7832
  AUC: 0.8425
  F1 分数: 0.7940
  MCC: 0.5674
  混淆矩阵:
[[418 145]
 [103 478]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/DT_mimic3_36_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/DT_mimic3_36_factors_notebook/metrics_summary.csv

决策树模型训练和评估完成。


In [17]:
# Cell 3: 定义训练和评估的主函数部分 (针对 MIMIC-IV 36因子数据)

# --- 手动设置参数 (模拟命令行参数) ---
class Args: # 创建一个简单的类来模拟 argparse 的 Namespace 对象
    pass

args = Args()

# === 配置你要测试的数据集和模型参数 ===
# 1. 指定预分割数据所在的目录
args.presplit_data_dir = './temp_data_utils_logs/mimic4_36_factors_split' # **** 修改这里 ****

# 2. 指定输出目录 (确保这个目录存在或脚本可以创建它)
#    为每个实验创建一个唯一的输出目录
data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
args.output_dir = f'./traditional_ml_outputs/DT_{data_source_tag.replace("_split", "")}_notebook' # **** 修改这里 ****

# 3. 标签列名
args.label_column = "dead"

# 4. 决策树参数 (与原始代码一致)
args.criterion = 'entropy'
args.splitter = 'random'

# 5. 标准化选项 (根据你的原始 stander_data 行为调整)
#    对于36因子数据，你需要确定原始代码是如何标准化的。
#    选项 A: 对所有36个特征进行标准化
standardize_all_features_flag = True
num_cols_to_standardize_value = None

#    选项 B: 只对前N列特征进行标准化 (你需要知道N是多少，例如原始的8或18)
#    如果你的原始代码对36因子数据只标准化了例如前18个特征：
# standardize_all_features_flag = False
# num_cols_to_standardize_value = 18 # **** 根据实际情况修改N ****

#    选项 C: 完全不进行标准化
# standardize_all_features_flag = False
# num_cols_to_standardize_value = None
# =======================================


# --- 开始执行训练和评估逻辑 ---
# (这部分代码与之前 Cell 3 的后半部分完全相同，无需修改)
os.makedirs(args.output_dir, exist_ok=True)
print(f"所有输出将保存到: {args.output_dir}")
print(f"使用的参数: {vars(args)}")


# 1. 加载和预处理数据
X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
    presplit_dir=args.presplit_data_dir,
    label_col=args.label_column,
    train_file="split_train_data_seed256.csv",
    test_file="split_test_data_seed256.csv",
    standardize_all_features=standardize_all_features_flag,
    num_cols_to_standardize=num_cols_to_standardize_value
)

# 2. 训练决策树模型
print(f"\n开始训练决策树模型: criterion='{args.criterion}', splitter='{args.splitter}'")
start_time = time.time()

base_dt_model = tree.DecisionTreeClassifier(
    criterion=args.criterion,
    splitter=args.splitter,
    random_state=256
)

calibrated_model = CalibratedClassifierCV(base_dt_model, method='isotonic', cv=3)
calibrated_model.fit(X_train, y_train)

model_save_path = os.path.join(args.output_dir, "decision_tree_calibrated.joblib")
joblib.dump(calibrated_model, model_save_path)
print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

# 3. 在测试集上评估并获取所需输出
y_pred_probs_test = calibrated_model.predict_proba(X_test)
y_pred_class_test = calibrated_model.predict(X_test)
positive_class_probs_test = y_pred_probs_test[:, 1]

test_acc = accuracy_score(y_test, y_pred_class_test)
test_auc = roc_auc_score(y_test, positive_class_probs_test)
test_f1 = f1_score(y_test, y_pred_class_test, average='binary')
test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
test_cm = confusion_matrix(y_test, y_pred_class_test)

print("\n--- 测试集评估结果 ---")
print(f"  准确率 (Accuracy): {test_acc:.4f}")
print(f"  AUC: {test_auc:.4f}")
print(f"  F1 分数: {test_f1:.4f}")
print(f"  MCC: {test_mcc:.4f}")
print(f"  混淆矩阵:\n{test_cm}")

# 4. 保存DeLong检验所需的文件
np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

# 5. 保存指标摘要
metrics_summary = {
    "model_name": "DecisionTree",
    "presplit_data_dir": args.presplit_data_dir,
    "criterion": args.criterion,
    "splitter": args.splitter,
    "test_accuracy": test_acc,
    "test_auc": test_auc,
    "test_f1": test_f1,
    "test_mcc": test_mcc,
    "confusion_matrix_tn": test_cm[0,0],
    "confusion_matrix_fp": test_cm[0,1],
    "confusion_matrix_fn": test_cm[1,0],
    "confusion_matrix_tp": test_cm[1,1],
}
metrics_df = pd.DataFrame([metrics_summary])
metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

print("\n决策树模型训练和评估完成。")

所有输出将保存到: ./traditional_ml_outputs/DT_mimic4_36_factors_notebook
使用的参数: {'presplit_data_dir': './temp_data_utils_logs/mimic4_36_factors_split', 'output_dir': './traditional_ml_outputs/DT_mimic4_36_factors_notebook', 'label_column': 'dead', 'criterion': 'entropy', 'splitter': 'random'}
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic4_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic4_36_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (4525, 36), 测试特征形状: (1940, 36)

开始训练决策树模型: criterion='entropy', splitter='random'
模型训练完成并在 0.080s 内保存至 ./traditional_ml_outputs/DT_mimic4_36_factors_notebook/decision_tree_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.8284
  AUC: 0.8831
  F1 分数: 0.8545
  MCC: 0.6691
  混淆矩阵:
[[629 279]
 [ 54 978]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/DT_mimic4_36_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/DT_mimic4_36_factors_notebook/metrics_summary.csv

决策树模型训练和评估完成。


In [22]:
# Cell 3: 定义训练和评估的主函数部分 (针对 MIMIC-III + eICU 8因子数据)

# --- 手动设置参数 (模拟命令行参数) ---
class Args: # 创建一个简单的类来模拟 argparse 的 Namespace 对象
    pass

args = Args()

# === 配置你要测试的数据集和模型参数 ===
# 1. 指定预分割数据所在的目录
args.presplit_data_dir = './temp_data_utils_logs/mimic3_eICU_8_factors_split' # **** 修改这里 ****

# 2. 指定输出目录 (确保这个目录存在或脚本可以创建它)
#    为每个实验创建一个唯一的输出目录
data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
args.output_dir = f'./traditional_ml_outputs/DT_{data_source_tag.replace("_split", "")}_notebook' # **** 修改这里 ****

# 3. 标签列名
args.label_column = "dead"

# 4. 决策树参数 (与原始代码一致)
args.criterion = 'entropy'
args.splitter = 'random'

# 5. 标准化选项
#    对于8因子数据，原始 stander_data 的逻辑是对所有8个输入特征进行标准化。
#    所以，我们在这里也对所有特征进行标准化。
standardize_all_features_flag = True
num_cols_to_standardize_value = None # 当 standardize_all_features_flag 为 True 时，此值不使用
# =======================================


# --- 开始执行训练和评估逻辑 ---
# (这部分代码与之前 Cell 3 的后半部分完全相同，无需修改)
os.makedirs(args.output_dir, exist_ok=True)
print(f"所有输出将保存到: {args.output_dir}")
print(f"使用的参数: {vars(args)}")


# 1. 加载和预处理数据
X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
    presplit_dir=args.presplit_data_dir,
    label_col=args.label_column,
    train_file="split_train_data_seed256.csv",
    test_file="split_test_data_seed256.csv",
    standardize_all_features=standardize_all_features_flag,
    num_cols_to_standardize=num_cols_to_standardize_value
)

# 2. 训练决策树模型
print(f"\n开始训练决策树模型: criterion='{args.criterion}', splitter='{args.splitter}'")
start_time = time.time()

base_dt_model = tree.DecisionTreeClassifier(
    criterion=args.criterion,
    splitter=args.splitter,
    random_state=256
)

calibrated_model = CalibratedClassifierCV(base_dt_model, method='isotonic', cv=2)
calibrated_model.fit(X_train, y_train)

model_save_path = os.path.join(args.output_dir, "decision_tree_calibrated.joblib")
joblib.dump(calibrated_model, model_save_path)
print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

# 3. 在测试集上评估并获取所需输出
y_pred_probs_test = calibrated_model.predict_proba(X_test)
y_pred_class_test = calibrated_model.predict(X_test)
positive_class_probs_test = y_pred_probs_test[:, 1]

test_acc = accuracy_score(y_test, y_pred_class_test)
test_auc = roc_auc_score(y_test, positive_class_probs_test)
test_f1 = f1_score(y_test, y_pred_class_test, average='binary')
test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
test_cm = confusion_matrix(y_test, y_pred_class_test)

print("\n--- 测试集评估结果 ---")
print(f"  准确率 (Accuracy): {test_acc:.4f}")
print(f"  AUC: {test_auc:.4f}")
print(f"  F1 分数: {test_f1:.4f}")
print(f"  MCC: {test_mcc:.4f}")
print(f"  混淆矩阵:\n{test_cm}")

# 4. 保存DeLong检验所需的文件
np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

# 5. 保存指标摘要
metrics_summary = {
    "model_name": "DecisionTree",
    "presplit_data_dir": args.presplit_data_dir,
    "criterion": args.criterion,
    "splitter": args.splitter,
    "test_accuracy": test_acc,
    "test_auc": test_auc,
    "test_f1": test_f1,
    "test_mcc": test_mcc,
    "confusion_matrix_tn": test_cm[0,0],
    "confusion_matrix_fp": test_cm[0,1],
    "confusion_matrix_fn": test_cm[1,0],
    "confusion_matrix_tp": test_cm[1,1],
}
metrics_df = pd.DataFrame([metrics_summary])
metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

print("\n决策树模型训练和评估完成。")

所有输出将保存到: ./traditional_ml_outputs/DT_mimic3_eICU_8_factors_notebook
使用的参数: {'presplit_data_dir': './temp_data_utils_logs/mimic3_eICU_8_factors_split', 'output_dir': './traditional_ml_outputs/DT_mimic3_eICU_8_factors_notebook', 'label_column': 'dead', 'criterion': 'entropy', 'splitter': 'random'}
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (8775, 8), 测试特征形状: (3762, 8)

开始训练决策树模型: criterion='entropy', splitter='random'
模型训练完成并在 0.046s 内保存至 ./traditional_ml_outputs/DT_mimic3_eICU_8_factors_notebook/decision_tree_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.7780
  AUC: 0.8359
  F1 分数: 0.7622
  MCC: 0.5620
  混淆矩阵:
[[1589  284]
 [ 551 1338]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/DT_mimic3_eICU_8_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/DT_mimic3_eICU_8_factors_notebook/metrics_sum

In [23]:
# Cell 3: 定义训练和评估的主函数部分 (针对 MIMIC-IV 8因子数据)

# --- 手动设置参数 (模拟命令行参数) ---
class Args: # 创建一个简单的类来模拟 argparse 的 Namespace 对象
    pass

args = Args()

# === 配置你要测试的数据集和模型参数 ===
# 1. 指定预分割数据所在的目录
args.presplit_data_dir = './temp_data_utils_logs/mimic4_8_factors_split' # **** 修改这里 ****

# 2. 指定输出目录 (确保这个目录存在或脚本可以创建它)
#    为每个实验创建一个唯一的输出目录
data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
args.output_dir = f'./traditional_ml_outputs/DT_{data_source_tag.replace("_split", "")}_notebook' # **** 修改这里 ****

# 3. 标签列名
args.label_column = "dead"

# 4. 决策树参数 (与原始代码一致)
args.criterion = 'entropy'
args.splitter = 'random'

# 5. 标准化选项
#    对于8因子数据，我们对所有8个输入特征进行标准化。
standardize_all_features_flag = True
num_cols_to_standardize_value = None # 当 standardize_all_features_flag 为 True 时，此值不使用
# =======================================


# --- 开始执行训练和评估逻辑 ---
# (这部分代码与之前 Cell 3 的后半部分完全相同，无需修改)
os.makedirs(args.output_dir, exist_ok=True)
print(f"所有输出将保存到: {args.output_dir}")
print(f"使用的参数: {vars(args)}")


# 1. 加载和预处理数据
X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
    presplit_dir=args.presplit_data_dir,
    label_col=args.label_column,
    train_file="split_train_data_seed256.csv",
    test_file="split_test_data_seed256.csv",
    standardize_all_features=standardize_all_features_flag,
    num_cols_to_standardize=num_cols_to_standardize_value
)

# 2. 训练决策树模型
print(f"\n开始训练决策树模型: criterion='{args.criterion}', splitter='{args.splitter}'")
start_time = time.time()

base_dt_model = tree.DecisionTreeClassifier(
    criterion=args.criterion,
    splitter=args.splitter,
    random_state=256
)

calibrated_model = CalibratedClassifierCV(base_dt_model, method='isotonic', cv=2)
calibrated_model.fit(X_train, y_train)

model_save_path = os.path.join(args.output_dir, "decision_tree_calibrated.joblib")
joblib.dump(calibrated_model, model_save_path)
print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

# 3. 在测试集上评估并获取所需输出
y_pred_probs_test = calibrated_model.predict_proba(X_test)
y_pred_class_test = calibrated_model.predict(X_test)
positive_class_probs_test = y_pred_probs_test[:, 1]

test_acc = accuracy_score(y_test, y_pred_class_test)
test_auc = roc_auc_score(y_test, positive_class_probs_test)
test_f1 = f1_score(y_test, y_pred_class_test, average='binary')
test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
test_cm = confusion_matrix(y_test, y_pred_class_test)

print("\n--- 测试集评估结果 ---")
print(f"  准确率 (Accuracy): {test_acc:.4f}")
print(f"  AUC: {test_auc:.4f}")
print(f"  F1 分数: {test_f1:.4f}")
print(f"  MCC: {test_mcc:.4f}")
print(f"  混淆矩阵:\n{test_cm}")

# 4. 保存DeLong检验所需的文件
np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

# 5. 保存指标摘要
metrics_summary = {
    "model_name": "DecisionTree",
    "presplit_data_dir": args.presplit_data_dir,
    "criterion": args.criterion,
    "splitter": args.splitter,
    "test_accuracy": test_acc,
    "test_auc": test_auc,
    "test_f1": test_f1,
    "test_mcc": test_mcc,
    "confusion_matrix_tn": test_cm[0,0],
    "confusion_matrix_fp": test_cm[0,1],
    "confusion_matrix_fn": test_cm[1,0],
    "confusion_matrix_tp": test_cm[1,1],
}
metrics_df = pd.DataFrame([metrics_summary])
metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

print("\n决策树模型训练和评估完成。")

所有输出将保存到: ./traditional_ml_outputs/DT_mimic4_8_factors_notebook
使用的参数: {'presplit_data_dir': './temp_data_utils_logs/mimic4_8_factors_split', 'output_dir': './traditional_ml_outputs/DT_mimic4_8_factors_notebook', 'label_column': 'dead', 'criterion': 'entropy', 'splitter': 'random'}
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (4525, 8), 测试特征形状: (1940, 8)

开始训练决策树模型: criterion='entropy', splitter='random'
模型训练完成并在 0.036s 内保存至 ./traditional_ml_outputs/DT_mimic4_8_factors_notebook/decision_tree_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.7108
  AUC: 0.7849
  F1 分数: 0.6906
  MCC: 0.4435
  混淆矩阵:
[[753 155]
 [406 626]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/DT_mimic4_8_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/DT_mimic4_8_factors_notebook/metrics_summary.csv

决策树模型训练和评估完成。


In [24]:
# Cell 3: 定义训练和评估的主函数部分 (针对 Local 8因子数据)

# --- 手动设置参数 (模拟命令行参数) ---
class Args: # 创建一个简单的类来模拟 argparse 的 Namespace 对象
    pass

args = Args()

# === 配置你要测试的数据集和模型参数 ===
# 1. 指定预分割数据所在的目录
args.presplit_data_dir = './temp_data_utils_logs/local_8_factors_split' # **** 修改这里 ****

# 2. 指定输出目录 (确保这个目录存在或脚本可以创建它)
#    为每个实验创建一个唯一的输出目录
data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
args.output_dir = f'./traditional_ml_outputs/DT_{data_source_tag.replace("_split", "")}_notebook' # **** 修改这里 ****

# 3. 标签列名
args.label_column = "dead" # 假设你的 local_8_factors_split 数据中标签列也叫 'dead'

# 4. 决策树参数 (与原始代码一致)
args.criterion = 'entropy'
args.splitter = 'random'

# 5. 标准化选项
#    对于8因子数据，我们对所有8个输入特征进行标准化。
standardize_all_features_flag = True
num_cols_to_standardize_value = None # 当 standardize_all_features_flag 为 True 时，此值不使用
# =======================================


# --- 开始执行训练和评估逻辑 ---
# (这部分代码与之前 Cell 3 的后半部分完全相同，无需修改)
os.makedirs(args.output_dir, exist_ok=True)
print(f"所有输出将保存到: {args.output_dir}")
print(f"使用的参数: {vars(args)}")


# 1. 加载和预处理数据
X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
    presplit_dir=args.presplit_data_dir,
    label_col=args.label_column,
    train_file="split_train_data_seed256.csv",
    test_file="split_test_data_seed256.csv",
    standardize_all_features=standardize_all_features_flag,
    num_cols_to_standardize=num_cols_to_standardize_value
)

# 2. 训练决策树模型
print(f"\n开始训练决策树模型: criterion='{args.criterion}', splitter='{args.splitter}'")
start_time = time.time()

base_dt_model = tree.DecisionTreeClassifier(
    criterion=args.criterion,
    splitter=args.splitter,
    random_state=256
)

calibrated_model = CalibratedClassifierCV(base_dt_model, method='isotonic', cv=2)
calibrated_model.fit(X_train, y_train)

model_save_path = os.path.join(args.output_dir, "decision_tree_calibrated.joblib")
joblib.dump(calibrated_model, model_save_path)
print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

# 3. 在测试集上评估并获取所需输出
y_pred_probs_test = calibrated_model.predict_proba(X_test)
y_pred_class_test = calibrated_model.predict(X_test)
positive_class_probs_test = y_pred_probs_test[:, 1]

test_acc = accuracy_score(y_test, y_pred_class_test)
test_auc = roc_auc_score(y_test, positive_class_probs_test)
test_f1 = f1_score(y_test, y_pred_class_test, average='binary')
test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
test_cm = confusion_matrix(y_test, y_pred_class_test)

print("\n--- 测试集评估结果 ---")
print(f"  准确率 (Accuracy): {test_acc:.4f}")
print(f"  AUC: {test_auc:.4f}")
print(f"  F1 分数: {test_f1:.4f}")
print(f"  MCC: {test_mcc:.4f}")
print(f"  混淆矩阵:\n{test_cm}")

# 4. 保存DeLong检验所需的文件
np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

# 5. 保存指标摘要
metrics_summary = {
    "model_name": "DecisionTree",
    "presplit_data_dir": args.presplit_data_dir,
    "criterion": args.criterion,
    "splitter": args.splitter,
    "test_accuracy": test_acc,
    "test_auc": test_auc,
    "test_f1": test_f1,
    "test_mcc": test_mcc,
    "confusion_matrix_tn": test_cm[0,0],
    "confusion_matrix_fp": test_cm[0,1],
    "confusion_matrix_fn": test_cm[1,0],
    "confusion_matrix_tp": test_cm[1,1],
}
metrics_df = pd.DataFrame([metrics_summary])
metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

print("\n决策树模型训练和评估完成。")

所有输出将保存到: ./traditional_ml_outputs/DT_local_8_factors_notebook
使用的参数: {'presplit_data_dir': './temp_data_utils_logs/local_8_factors_split', 'output_dir': './traditional_ml_outputs/DT_local_8_factors_notebook', 'label_column': 'dead', 'criterion': 'entropy', 'splitter': 'random'}
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/local_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/local_8_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (435, 8), 测试特征形状: (187, 8)

开始训练决策树模型: criterion='entropy', splitter='random'
模型训练完成并在 0.016s 内保存至 ./traditional_ml_outputs/DT_local_8_factors_notebook/decision_tree_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.7005
  AUC: 0.7120
  F1 分数: 0.6706
  MCC: 0.4335
  混淆矩阵:
[[74 13]
 [43 57]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/DT_local_8_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/DT_local_8_factors_notebook/metrics_summary.csv

决策树模型训练和评估完成。


In [25]:
import os
import time
import pandas as pd
import numpy as np
import joblib
# from sklearn import tree # 不需要决策树了
from sklearn.linear_model import LogisticRegression # 导入逻辑回归
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import StandardScaler
# from data_utils import CustomDataset, load_presplit_data_from_dir

In [26]:
def load_and_preprocess_data(presplit_dir, label_col, train_file, test_file, standardize_all_features=True, num_cols_to_standardize=None):
    # ... (函数体与之前完全相同) ...
    train_file_path = os.path.join(presplit_dir, train_file)
    test_file_path = os.path.join(presplit_dir, test_file)
    if not os.path.exists(train_file_path):
        raise FileNotFoundError(f"预分割的训练数据文件未找到: {train_file_path}")
    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"预分割的测试数据文件未找到: {test_file_path}")
    print(f"从预分割文件加载数据:")
    print(f"  训练数据: {train_file_path}")
    print(f"  测试数据: {test_file_path}")
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)
    if label_col not in train_df.columns or label_col not in test_df.columns:
        raise ValueError(f"标签列 '{label_col}' 未在预分割的CSV文件中找到。")
    X_train_raw = train_df.drop(label_col, axis=1)
    y_train = train_df[label_col].to_numpy()
    X_test_raw = test_df.drop(label_col, axis=1)
    y_test = test_df[label_col].to_numpy()
    feature_names = X_train_raw.columns.tolist()
    if standardize_all_features:
        print("对所有特征列进行标准化 (基于训练集拟合)...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)
    elif num_cols_to_standardize is not None and num_cols_to_standardize > 0:
        print(f"将对前 {num_cols_to_standardize} 列特征进行标准化 (基于训练集拟合)...")
        X_train_to_scale = X_train_raw.iloc[:, :num_cols_to_standardize]
        X_train_not_scaled = X_train_raw.iloc[:, num_cols_to_standardize:]
        X_test_to_scale = X_test_raw.iloc[:, :num_cols_to_standardize]
        X_test_not_scaled = X_test_raw.iloc[:, num_cols_to_standardize:]
        scaler = StandardScaler()
        X_train_scaled_part = scaler.fit_transform(X_train_to_scale)
        X_test_scaled_part = scaler.transform(X_test_to_scale)
        if isinstance(X_train_not_scaled, pd.DataFrame):
             X_train_scaled = pd.concat([pd.DataFrame(X_train_scaled_part, columns=X_train_to_scale.columns, index=X_train_to_scale.index),
                                      X_train_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
             X_test_scaled = pd.concat([pd.DataFrame(X_test_scaled_part, columns=X_test_to_scale.columns, index=X_test_to_scale.index),
                                     X_test_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
        else:
            X_train_scaled = np.hstack((X_train_scaled_part, X_train_not_scaled.to_numpy() if hasattr(X_train_not_scaled, 'to_numpy') else X_train_not_scaled))
            X_test_scaled = np.hstack((X_test_scaled_part, X_test_not_scaled.to_numpy() if hasattr(X_test_not_scaled, 'to_numpy') else X_test_not_scaled))
    else:
        print("不进行特征标准化处理。")
        X_train_scaled = X_train_raw.to_numpy()
        X_test_scaled = X_test_raw.to_numpy()
    print(f"数据加载和预处理完成。训练特征形状: {X_train_scaled.shape}, 测试特征形状: {X_test_scaled.shape}")
    return X_train_scaled, y_train, X_test_scaled, y_test, feature_names

In [27]:
# Cell 3: 训练和评估逻辑回归 (在所有指定数据集上)

# --- 手动设置参数 ---
class Args: # 创建一个简单的类来模拟 argparse 的 Namespace 对象
    pass

# === 定义所有要测试的数据集 ===
# 格式: "dataset_tag": {standardize_all: True/False, num_cols_to_standardize: None or int}
# standardize_all: True -> 对所有特征标准化
# standardize_all: False, num_cols_to_standardize: N -> 只对前N列标准化
# standardize_all: False, num_cols_to_standardize: None -> 不标准化
datasets_to_process = {
    "mimic3_36_factors": {"standardize_all": True, "num_cols_to_standardize": None}, # 假设对所有36列标准化
    "mimic4_36_factors": {"standardize_all": True, "num_cols_to_standardize": None}, # 假设对所有36列标准化
    "mimic3_eICU_8_factors": {"standardize_all": True, "num_cols_to_standardize": None}, # 8因子数据，对所有8列标准化
    "mimic4_8_factors": {"standardize_all": True, "num_cols_to_standardize": None},    # 8因子数据，对所有8列标准化
    "local_8_factors": {"standardize_all": True, "num_cols_to_standardize": None}     # 8因子数据，对所有8列标准化
}
# !!! 重要: 对于36因子数据，如果原始 stander_data 只标准化了部分列 (例如前8或18列)，
# 你需要将对应数据集的 "standardize_all" 改为 False，并设置 "num_cols_to_standardize" 为正确的列数。
# 例如:
# "mimic3_36_factors": {"standardize_all": False, "num_cols_to_standardize": 18},
# =====================================

# --- 主循环，遍历所有数据集 ---
for dataset_tag, std_config in datasets_to_process.items():
    print(f"\n\n{'='*30} PROCESSING DATASET: {dataset_tag} {'='*30}")
    args = Args()

    # 1. 指定预分割数据所在的目录
    args.presplit_data_dir = f'./temp_data_utils_logs/{dataset_tag}_split'

    # 2. 指定输出目录
    args.output_dir = f'./traditional_ml_outputs/LR_{dataset_tag}_notebook' # LR for Logistic Regression

    # 3. 标签列名
    args.label_column = "dead"

    # --- 标准化选项 ---
    standardize_all_features_flag = std_config["standardize_all"]
    num_cols_to_standardize_value = std_config["num_cols_to_standardize"]
    # --------------------

    # --- 开始执行当前数据集的训练和评估逻辑 ---
    os.makedirs(args.output_dir, exist_ok=True)
    print(f"输出将保存到: {args.output_dir}")
    # print(f"参数: {vars(args)}") # vars(args) 在这个简单类上可能不直接工作，但我们知道值

    # 1. 加载和预处理数据
    print(f"\n--- 加载数据 for {dataset_tag} ---")
    try:
        X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
            presplit_dir=args.presplit_data_dir,
            label_col=args.label_column,
            train_file="split_train_data_seed256.csv",
            test_file="split_test_data_seed256.csv",
            standardize_all_features=standardize_all_features_flag,
            num_cols_to_standardize=num_cols_to_standardize_value
        )
    except FileNotFoundError as e:
        print(f"错误: 数据文件未找到 for {dataset_tag} at {args.presplit_data_dir}. 跳过此数据集。错误信息: {e}")
        continue # 跳到下一个数据集

    # 2. 训练逻辑回归模型
    print(f"\n--- 开始训练逻辑回归模型 for {dataset_tag} ---")
    start_time = time.time()

    base_lr_model = LogisticRegression(
        random_state=256,
        solver='liblinear', # 或 'lbfgs' 等
        max_iter=1000      # 增加迭代次数确保收敛
        # penalty='l2', C=1.0 # 可以显式设置默认值
    )

    calibrated_model = CalibratedClassifierCV(base_lr_model, method='isotonic', cv=5)
    calibrated_model.fit(X_train, y_train)

    model_save_path = os.path.join(args.output_dir, "logistic_regression_calibrated.joblib")
    joblib.dump(calibrated_model, model_save_path)
    print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

    # 3. 在测试集上评估并获取所需输出
    y_pred_probs_test = calibrated_model.predict_proba(X_test)
    y_pred_class_test = calibrated_model.predict(X_test)
    positive_class_probs_test = y_pred_probs_test[:, 1]

    test_acc = accuracy_score(y_test, y_pred_class_test)
    # 确保至少有两个类别才能计算AUC
    if len(np.unique(y_test)) > 1:
        test_auc = roc_auc_score(y_test, positive_class_probs_test)
    else:
        print(f"警告: 数据集 {dataset_tag} 的测试集只包含一个类别，AUC无法计算，设为0.0。")
        test_auc = 0.0
        
    test_f1 = f1_score(y_test, y_pred_class_test, average='binary', zero_division=0)
    test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
    test_cm = confusion_matrix(y_test, y_pred_class_test)

    print("\n--- 测试集评估结果 ---")
    print(f"  准确率 (Accuracy): {test_acc:.4f}")
    print(f"  AUC: {test_auc:.4f}")
    print(f"  F1 分数: {test_f1:.4f}")
    print(f"  MCC: {test_mcc:.4f}")
    print(f"  混淆矩阵:\n{test_cm}")

    # 4. 保存DeLong检验所需的文件
    np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
    np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
    print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

    # 5. 保存指标摘要
    tn_val, fp_val, fn_val, tp_val = (0,0,0,0)
    if test_cm.size == 4: # 确保是2x2矩阵
        tn_val, fp_val, fn_val, tp_val = test_cm.ravel()
    elif test_cm.size == 1 and len(y_test) > 0: # 处理所有样本都是一个类别且预测正确的情况
        if y_test[0] == 0 and y_pred_class_test[0] == 0 : tn_val = test_cm[0,0]
        elif y_test[0] == 1 and y_pred_class_test[0] == 1 : tp_val = test_cm[0,0]

    metrics_summary = {
        "model_name": "LogisticRegression",
        "dataset_tag": dataset_tag, # 添加数据集标识
        "presplit_data_dir": args.presplit_data_dir,
        "test_accuracy": test_acc,
        "test_auc": test_auc,
        "test_f1": test_f1,
        "test_mcc": test_mcc,
        "confusion_matrix_tn": tn_val,
        "confusion_matrix_fp": fp_val,
        "confusion_matrix_fn": fn_val,
        "confusion_matrix_tp": tp_val,
    }
    metrics_df = pd.DataFrame([metrics_summary])
    metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
    print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

    print(f"\n逻辑回归模型 ({dataset_tag}) 训练和评估完成。")

print(f"\n\n{'='*30} 所有数据集处理完毕 {'='*30}")



============================== PROCESSING DATASET: mimic3_36_factors ==============================
输出将保存到: ./traditional_ml_outputs/LR_mimic3_36_factors_notebook

--- 加载数据 for mimic3_36_factors ---
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (2667, 36), 测试特征形状: (1144, 36)

--- 开始训练逻辑回归模型 for mimic3_36_factors ---
模型训练完成并在 0.133s 内保存至 ./traditional_ml_outputs/LR_mimic3_36_factors_notebook/logistic_regression_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.7194
  AUC: 0.7896
  F1 分数: 0.7268
  MCC: 0.4386
  混淆矩阵:
[[396 167]
 [154 427]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/LR_mimic3_36_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/LR_mimic3_36_factors_notebook/metrics_summary.csv

逻辑回归模型 (mimic3_36_factors) 训练和评估完成。


============================== PROCESSING DATASET: mimic4_36_factors =====

In [28]:
import os
import time
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier # 导入随机森林
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import StandardScaler
# from data_utils import CustomDataset, load_presplit_data_from_dir

In [29]:
def load_and_preprocess_data(presplit_dir, label_col, train_file, test_file, standardize_all_features=True, num_cols_to_standardize=None):
    # ... (函数体与之前完全相同) ...
    train_file_path = os.path.join(presplit_dir, train_file)
    test_file_path = os.path.join(presplit_dir, test_file)
    if not os.path.exists(train_file_path):
        raise FileNotFoundError(f"预分割的训练数据文件未找到: {train_file_path}")
    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"预分割的测试数据文件未找到: {test_file_path}")
    print(f"从预分割文件加载数据:")
    print(f"  训练数据: {train_file_path}")
    print(f"  测试数据: {test_file_path}")
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)
    if label_col not in train_df.columns or label_col not in test_df.columns:
        raise ValueError(f"标签列 '{label_col}' 未在预分割的CSV文件中找到。")
    X_train_raw = train_df.drop(label_col, axis=1)
    y_train = train_df[label_col].to_numpy()
    X_test_raw = test_df.drop(label_col, axis=1)
    y_test = test_df[label_col].to_numpy()
    feature_names = X_train_raw.columns.tolist()
    if standardize_all_features:
        print("对所有特征列进行标准化 (基于训练集拟合)...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)
    elif num_cols_to_standardize is not None and num_cols_to_standardize > 0:
        print(f"将对前 {num_cols_to_standardize} 列特征进行标准化 (基于训练集拟合)...")
        X_train_to_scale = X_train_raw.iloc[:, :num_cols_to_standardize]
        X_train_not_scaled = X_train_raw.iloc[:, num_cols_to_standardize:]
        X_test_to_scale = X_test_raw.iloc[:, :num_cols_to_standardize]
        X_test_not_scaled = X_test_raw.iloc[:, num_cols_to_standardize:]
        scaler = StandardScaler()
        X_train_scaled_part = scaler.fit_transform(X_train_to_scale)
        X_test_scaled_part = scaler.transform(X_test_to_scale)
        if isinstance(X_train_not_scaled, pd.DataFrame):
             X_train_scaled = pd.concat([pd.DataFrame(X_train_scaled_part, columns=X_train_to_scale.columns, index=X_train_to_scale.index),
                                      X_train_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
             X_test_scaled = pd.concat([pd.DataFrame(X_test_scaled_part, columns=X_test_to_scale.columns, index=X_test_to_scale.index),
                                     X_test_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
        else:
            X_train_scaled = np.hstack((X_train_scaled_part, X_train_not_scaled.to_numpy() if hasattr(X_train_not_scaled, 'to_numpy') else X_train_not_scaled))
            X_test_scaled = np.hstack((X_test_scaled_part, X_test_not_scaled.to_numpy() if hasattr(X_test_not_scaled, 'to_numpy') else X_test_not_scaled))
    else:
        print("不进行特征标准化处理。")
        X_train_scaled = X_train_raw.to_numpy()
        X_test_scaled = X_test_raw.to_numpy()
    print(f"数据加载和预处理完成。训练特征形状: {X_train_scaled.shape}, 测试特征形状: {X_test_scaled.shape}")
    return X_train_scaled, y_train, X_test_scaled, y_test, feature_names

In [37]:
# Cell 3: 训练和评估随机森林 (模板 - 为每个数据集修改参数)

# --- 手动设置参数 ---
class Args:
    pass

# === 定义所有要测试的数据集 (与逻辑回归的定义相同) ===
datasets_to_process = {
    "mimic3_36_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic4_36_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic3_eICU_8_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic4_8_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "local_8_factors": {"standardize_all": True, "num_cols_to_standardize": None}
}
# !!! 重要: 对于36因子数据，如果原始 stander_data 只标准化了部分列，请修改上面的配置。
# 例如: "mimic3_36_factors": {"standardize_all": False, "num_cols_to_standardize": 18}
# =============================================================================

# --- 主循环，遍历所有数据集 ---
for dataset_tag, std_config in datasets_to_process.items():
    print(f"\n\n{'='*30} PROCESSING DATASET (Random Forest): {dataset_tag} {'='*30}")
    args = Args()

    # 1. 指定预分割数据所在的目录
    args.presplit_data_dir = f'./temp_data_utils_logs/{dataset_tag}_split'

    # 2. 指定输出目录
    args.output_dir = f'./traditional_ml_outputs/RF_{dataset_tag}_notebook' # RF for Random Forest

    # 3. 标签列名
    args.label_column = "dead"

    # --- 随机森林参数 (基于你的原始代码) ---
    args.rf_n_estimators = 13  # 原始代码 RF_classifier 函数的默认值
    args.rf_max_depth = 8   # 原始代码 RF_classifier 函数的默认值
    # 你可以根据需要为不同数据集或实验修改这些值
    # 例如: if dataset_tag == "mimic3_36_factors": args.rf_n_estimators = 20
    # -----------------------------------------

    # --- 标准化选项 ---
    standardize_all_features_flag = std_config["standardize_all"]
    num_cols_to_standardize_value = std_config["num_cols_to_standardize"]
    # --------------------


    # --- 开始执行当前数据集的训练和评估逻辑 ---
    os.makedirs(args.output_dir, exist_ok=True)
    print(f"输出将保存到: {args.output_dir}")
    # print(f"参数: {vars(args)}")

    # 1. 加载和预处理数据
    print(f"\n--- 加载数据 for {dataset_tag} ---")
    try:
        X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
            presplit_dir=args.presplit_data_dir,
            label_col=args.label_column,
            train_file="split_train_data_seed256.csv",
            test_file="split_test_data_seed256.csv",
            standardize_all_features=standardize_all_features_flag,
            num_cols_to_standardize=num_cols_to_standardize_value
        )
    except FileNotFoundError as e:
        print(f"错误: 数据文件未找到 for {dataset_tag} at {args.presplit_data_dir}. 跳过此数据集。错误信息: {e}")
        continue # 跳到下一个数据集

    # 2. 训练随机森林模型
    print(f"\n--- 开始训练随机森林模型 for {dataset_tag} ---")
    print(f"  n_estimators: {args.rf_n_estimators}, max_depth: {args.rf_max_depth}")
    start_time = time.time()

    base_rf_model = RandomForestClassifier(
        n_estimators=args.rf_n_estimators,
        max_depth=args.rf_max_depth,
        random_state=0, # 与原始代码一致
        n_jobs=-1       # 使用所有可用核心并行处理，可以加速训练
    )

    # 使用 CalibratedClassifierCV 进行概率校准
    calibrated_model = CalibratedClassifierCV(base_rf_model, method='isotonic', cv=2)
    calibrated_model.fit(X_train, y_train)

    model_save_path = os.path.join(args.output_dir, "random_forest_calibrated.joblib")
    joblib.dump(calibrated_model, model_save_path)
    print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

    # 3. 在测试集上评估并获取所需输出
    y_pred_probs_test = calibrated_model.predict_proba(X_test)
    y_pred_class_test = calibrated_model.predict(X_test)
    positive_class_probs_test = y_pred_probs_test[:, 1]

    test_acc = accuracy_score(y_test, y_pred_class_test)
    if len(np.unique(y_test)) > 1:
        test_auc = roc_auc_score(y_test, positive_class_probs_test)
    else:
        print(f"警告: 数据集 {dataset_tag} 的测试集只包含一个类别，AUC无法计算，设为0.0。")
        test_auc = 0.0
    test_f1 = f1_score(y_test, y_pred_class_test, average='binary', zero_division=0)
    test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
    test_cm = confusion_matrix(y_test, y_pred_class_test)

    print("\n--- 测试集评估结果 ---")
    print(f"  准确率 (Accuracy): {test_acc:.4f}")
    print(f"  AUC: {test_auc:.4f}")
    print(f"  F1 分数: {test_f1:.4f}")
    print(f"  MCC: {test_mcc:.4f}")
    print(f"  混淆矩阵:\n{test_cm}")

    # 4. 保存DeLong检验所需的文件
    np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
    np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
    print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

    # 5. 保存指标摘要
    tn_val, fp_val, fn_val, tp_val = (0,0,0,0)
    if test_cm.size == 4: tn_val, fp_val, fn_val, tp_val = test_cm.ravel()
    elif test_cm.size == 1 and len(y_test) > 0:
        if y_test[0] == 0 and y_pred_class_test[0] == 0 : tn_val = test_cm[0,0]
        elif y_test[0] == 1 and y_pred_class_test[0] == 1 : tp_val = test_cm[0,0]

    metrics_summary = {
        "model_name": "RandomForest", # 修改模型名称
        "dataset_tag": dataset_tag,
        "presplit_data_dir": args.presplit_data_dir,
        "rf_n_estimators": args.rf_n_estimators,
        "rf_max_depth": args.rf_max_depth if args.rf_max_depth is not None else "None",
        "test_accuracy": test_acc,
        "test_auc": test_auc,
        "test_f1": test_f1,
        "test_mcc": test_mcc,
        "confusion_matrix_tn": tn_val,
        "confusion_matrix_fp": fp_val,
        "confusion_matrix_fn": fn_val,
        "confusion_matrix_tp": tp_val,
    }
    metrics_df = pd.DataFrame([metrics_summary])
    metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
    print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

    print(f"\n随机森林模型 ({dataset_tag}) 训练和评估完成。")

print(f"\n\n{'='*30} 所有数据集处理完毕 (Random Forest) {'='*30}")



============================== PROCESSING DATASET (Random Forest): mimic3_36_factors ==============================
输出将保存到: ./traditional_ml_outputs/RF_mimic3_36_factors_notebook

--- 加载数据 for mimic3_36_factors ---
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (2667, 36), 测试特征形状: (1144, 36)

--- 开始训练随机森林模型 for mimic3_36_factors ---
  n_estimators: 13, max_depth: 8
模型训练完成并在 0.209s 内保存至 ./traditional_ml_outputs/RF_mimic3_36_factors_notebook/random_forest_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.8121
  AUC: 0.8850
  F1 分数: 0.8167
  MCC: 0.6240
  混淆矩阵:
[[450 113]
 [102 479]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/RF_mimic3_36_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/RF_mimic3_36_factors_notebook/metrics_summary.csv

随机森林模型 (mimic3_36_factors) 训练和评估完成。


============================== 

In [38]:
import os
import time
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import SGDClassifier # 导入SGD分类器
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import StandardScaler
# from data_utils import CustomDataset, load_presplit_data_from_dir
def load_and_preprocess_data(presplit_dir, label_col, train_file, test_file, standardize_all_features=True, num_cols_to_standardize=None):
    # ... (函数体与之前完全相同) ...
    train_file_path = os.path.join(presplit_dir, train_file)
    test_file_path = os.path.join(presplit_dir, test_file)
    if not os.path.exists(train_file_path):
        raise FileNotFoundError(f"预分割的训练数据文件未找到: {train_file_path}")
    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"预分割的测试数据文件未找到: {test_file_path}")
    print(f"从预分割文件加载数据:")
    print(f"  训练数据: {train_file_path}")
    print(f"  测试数据: {test_file_path}")
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)
    if label_col not in train_df.columns or label_col not in test_df.columns:
        raise ValueError(f"标签列 '{label_col}' 未在预分割的CSV文件中找到。")
    X_train_raw = train_df.drop(label_col, axis=1)
    y_train = train_df[label_col].to_numpy()
    X_test_raw = test_df.drop(label_col, axis=1)
    y_test = test_df[label_col].to_numpy()
    feature_names = X_train_raw.columns.tolist()
    if standardize_all_features:
        print("对所有特征列进行标准化 (基于训练集拟合)...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)
    elif num_cols_to_standardize is not None and num_cols_to_standardize > 0:
        print(f"将对前 {num_cols_to_standardize} 列特征进行标准化 (基于训练集拟合)...")
        X_train_to_scale = X_train_raw.iloc[:, :num_cols_to_standardize]
        X_train_not_scaled = X_train_raw.iloc[:, num_cols_to_standardize:]
        X_test_to_scale = X_test_raw.iloc[:, :num_cols_to_standardize]
        X_test_not_scaled = X_test_raw.iloc[:, num_cols_to_standardize:]
        scaler = StandardScaler()
        X_train_scaled_part = scaler.fit_transform(X_train_to_scale)
        X_test_scaled_part = scaler.transform(X_test_to_scale)
        if isinstance(X_train_not_scaled, pd.DataFrame):
             X_train_scaled = pd.concat([pd.DataFrame(X_train_scaled_part, columns=X_train_to_scale.columns, index=X_train_to_scale.index),
                                      X_train_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
             X_test_scaled = pd.concat([pd.DataFrame(X_test_scaled_part, columns=X_test_to_scale.columns, index=X_test_to_scale.index),
                                     X_test_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
        else:
            X_train_scaled = np.hstack((X_train_scaled_part, X_train_not_scaled.to_numpy() if hasattr(X_train_not_scaled, 'to_numpy') else X_train_not_scaled))
            X_test_scaled = np.hstack((X_test_scaled_part, X_test_not_scaled.to_numpy() if hasattr(X_test_not_scaled, 'to_numpy') else X_test_not_scaled))
    else:
        print("不进行特征标准化处理。")
        X_train_scaled = X_train_raw.to_numpy()
        X_test_scaled = X_test_raw.to_numpy()
    print(f"数据加载和预处理完成。训练特征形状: {X_train_scaled.shape}, 测试特征形状: {X_test_scaled.shape}")
    return X_train_scaled, y_train, X_test_scaled, y_test, feature_names
# Cell 3: 训练和评估SGDClassifier (模板 - 为每个数据集修改参数)

# --- 手动设置参数 ---
class Args:
    pass

# === 定义所有要测试的数据集 (与之前的定义相同) ===
datasets_to_process = {
    "mimic3_36_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic4_36_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic3_eICU_8_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic4_8_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "local_8_factors": {"standardize_all": True, "num_cols_to_standardize": None}
}
# !!! 重要: 对于36因子数据，如果原始 stander_data 只标准化了部分列，请修改上面的配置。
# =============================================================================

# --- 主循环，遍历所有数据集 ---
for dataset_tag, std_config in datasets_to_process.items():
    print(f"\n\n{'='*30} PROCESSING DATASET (SGDClassifier): {dataset_tag} {'='*30}")
    args = Args()

    # 1. 指定预分割数据所在的目录
    args.presplit_data_dir = f'./temp_data_utils_logs/{dataset_tag}_split'

    # 2. 指定输出目录
    args.output_dir = f'./traditional_ml_outputs/SGD_{dataset_tag}_notebook' # SGD for SGDClassifier

    # 3. 标签列名
    args.label_column = "dead"

    # --- SGDClassifier 参数 (基于你的原始代码) ---
    # 你的原始 main() 中设置了 loss = 'hinge', penalty = 'l2'
    args.sgd_loss = 'hinge'    # 可选: 'hinge', 'log_loss' (与log相同效果), 'modified_huber', 'squared_hinge', 'perceptron', etc.
                               # 'log_loss' 会让它表现得像逻辑回归
    args.sgd_penalty = 'l2'    # 可选: 'l1', 'l2', 'elasticnet'
    args.sgd_max_iter = 2000   # SGD可能需要更多迭代次数才能收敛，原始默认是1000
    args.sgd_tol = 1e-3        # 收敛的容忍度
    args.sgd_alpha = 0.0001    # 正则化强度 (默认值)
    # 你可以根据需要为不同数据集或实验修改这些值
    # -----------------------------------------

    # --- 标准化选项 ---
    standardize_all_features_flag = std_config["standardize_all"]
    num_cols_to_standardize_value = std_config["num_cols_to_standardize"]
    # --------------------

    # --- 开始执行当前数据集的训练和评估逻辑 ---
    os.makedirs(args.output_dir, exist_ok=True)
    print(f"输出将保存到: {args.output_dir}")
    # print(f"参数: {vars(args)}")

    # 1. 加载和预处理数据
    print(f"\n--- 加载数据 for {dataset_tag} ---")
    try:
        X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
            presplit_dir=args.presplit_data_dir,
            label_col=args.label_column,
            train_file="split_train_data_seed256.csv",
            test_file="split_test_data_seed256.csv",
            standardize_all_features=standardize_all_features_flag,
            num_cols_to_standardize=num_cols_to_standardize_value
        )
    except FileNotFoundError as e:
        print(f"错误: 数据文件未找到 for {dataset_tag} at {args.presplit_data_dir}. 跳过此数据集。错误信息: {e}")
        continue # 跳到下一个数据集
    except ValueError as e: # 捕获 load_and_preprocess_data 中可能因标签列引发的 ValueError
        print(f"错误: 加载数据时发生错误 for {dataset_tag}. 跳过此数据集。错误信息: {e}")
        continue


    # 2. 训练 SGDClassifier 模型
    print(f"\n--- 开始训练 SGDClassifier for {dataset_tag} ---")
    print(f"  loss: {args.sgd_loss}, penalty: {args.sgd_penalty}, max_iter: {args.sgd_max_iter}")
    start_time = time.time()

    # SGDClassifier 对特征缩放敏感，确保数据已标准化
    base_sgd_model = SGDClassifier(
        loss=args.sgd_loss,
        penalty=args.sgd_penalty,
        alpha=args.sgd_alpha,
        max_iter=args.sgd_max_iter,
        tol=args.sgd_tol,
        random_state=256, # 为了可复现性
        n_jobs=-1,        # 使用所有可用核心
        learning_rate='optimal' # 默认值，通常不错
    )

    # 使用 CalibratedClassifierCV 进行概率校准
    # 注意：CalibratedClassifierCV 的 base_estimator 不应该已经 fit 过，除非 method='prefit'
    # SGDClassifier 本身可以输出概率（如果 loss='log_loss' or 'modified_huber'），
    # 但为了与你的原始流程一致，我们仍然使用 CalibratedClassifierCV
    # 对于 'hinge' loss (线性SVM)，它本身不直接输出概率，所以校准是必要的。
    calibrated_model = CalibratedClassifierCV(base_sgd_model, method='isotonic', cv=5)
    calibrated_model.fit(X_train, y_train)

    model_save_path = os.path.join(args.output_dir, "sgd_classifier_calibrated.joblib")
    joblib.dump(calibrated_model, model_save_path)
    print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

    # 3. 在测试集上评估并获取所需输出
    y_pred_probs_test = calibrated_model.predict_proba(X_test)
    y_pred_class_test = calibrated_model.predict(X_test)
    positive_class_probs_test = y_pred_probs_test[:, 1]

    test_acc = accuracy_score(y_test, y_pred_class_test)
    if len(np.unique(y_test)) > 1:
        test_auc = roc_auc_score(y_test, positive_class_probs_test)
    else:
        print(f"警告: 数据集 {dataset_tag} 的测试集只包含一个类别，AUC无法计算，设为0.0。")
        test_auc = 0.0
    test_f1 = f1_score(y_test, y_pred_class_test, average='binary', zero_division=0)
    test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
    test_cm = confusion_matrix(y_test, y_pred_class_test)

    print("\n--- 测试集评估结果 ---")
    print(f"  准确率 (Accuracy): {test_acc:.4f}")
    print(f"  AUC: {test_auc:.4f}")
    print(f"  F1 分数: {test_f1:.4f}")
    print(f"  MCC: {test_mcc:.4f}")
    print(f"  混淆矩阵:\n{test_cm}")

    # 4. 保存DeLong检验所需的文件
    np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
    np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
    print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

    # 5. 保存指标摘要
    tn_val, fp_val, fn_val, tp_val = (0,0,0,0)
    if test_cm.size == 4: tn_val, fp_val, fn_val, tp_val = test_cm.ravel()
    elif test_cm.size == 1 and len(y_test) > 0:
        if y_test[0] == 0 and y_pred_class_test[0] == 0 : tn_val = test_cm[0,0]
        elif y_test[0] == 1 and y_pred_class_test[0] == 1 : tp_val = test_cm[0,0]

    metrics_summary = {
        "model_name": "SGDClassifier", # 修改模型名称
        "dataset_tag": dataset_tag,
        "presplit_data_dir": args.presplit_data_dir,
        "sgd_loss": args.sgd_loss,
        "sgd_penalty": args.sgd_penalty,
        "sgd_max_iter": args.sgd_max_iter,
        "test_accuracy": test_acc,
        "test_auc": test_auc,
        "test_f1": test_f1,
        "test_mcc": test_mcc,
        "confusion_matrix_tn": tn_val,
        "confusion_matrix_fp": fp_val,
        "confusion_matrix_fn": fn_val,
        "confusion_matrix_tp": tp_val,
    }
    metrics_df = pd.DataFrame([metrics_summary])
    metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
    print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

    print(f"\nSGDClassifier模型 ({dataset_tag}) 训练和评估完成。")

print(f"\n\n{'='*30} 所有数据集处理完毕 (SGDClassifier) {'='*30}")



============================== PROCESSING DATASET (SGDClassifier): mimic3_36_factors ==============================
输出将保存到: ./traditional_ml_outputs/SGD_mimic3_36_factors_notebook

--- 加载数据 for mimic3_36_factors ---
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (2667, 36), 测试特征形状: (1144, 36)

--- 开始训练 SGDClassifier for mimic3_36_factors ---
  loss: hinge, penalty: l2, max_iter: 2000
模型训练完成并在 0.194s 内保存至 ./traditional_ml_outputs/SGD_mimic3_36_factors_notebook/sgd_classifier_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.6993
  AUC: 0.7674
  F1 分数: 0.7239
  MCC: 0.4014
  混淆矩阵:
[[349 214]
 [130 451]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/SGD_mimic3_36_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/SGD_mimic3_36_factors_notebook/metrics_summary.csv

SGDClassifier模型 (mimic3_36_factors) 训练和评估完成。



In [39]:
import os
import time
import pandas as pd
import numpy as np
import joblib
import sklearn.svm # 导入SVM
import sklearn.calibration # 确保导入
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import StandardScaler
# from data_utils import CustomDataset, load_presplit_data_from_dir
def load_and_preprocess_data(presplit_dir, label_col, train_file, test_file, standardize_all_features=True, num_cols_to_standardize=None):
    # ... (函数体与之前完全相同) ...
    train_file_path = os.path.join(presplit_dir, train_file)
    test_file_path = os.path.join(presplit_dir, test_file)
    if not os.path.exists(train_file_path):
        raise FileNotFoundError(f"预分割的训练数据文件未找到: {train_file_path}")
    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"预分割的测试数据文件未找到: {test_file_path}")
    print(f"从预分割文件加载数据:")
    print(f"  训练数据: {train_file_path}")
    print(f"  测试数据: {test_file_path}")
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)
    if label_col not in train_df.columns or label_col not in test_df.columns:
        raise ValueError(f"标签列 '{label_col}' 未在预分割的CSV文件中找到。")
    X_train_raw = train_df.drop(label_col, axis=1)
    y_train = train_df[label_col].to_numpy()
    X_test_raw = test_df.drop(label_col, axis=1)
    y_test = test_df[label_col].to_numpy()
    feature_names = X_train_raw.columns.tolist()
    if standardize_all_features:
        print("对所有特征列进行标准化 (基于训练集拟合)...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)
    elif num_cols_to_standardize is not None and num_cols_to_standardize > 0:
        print(f"将对前 {num_cols_to_standardize} 列特征进行标准化 (基于训练集拟合)...")
        X_train_to_scale = X_train_raw.iloc[:, :num_cols_to_standardize]
        X_train_not_scaled = X_train_raw.iloc[:, num_cols_to_standardize:]
        X_test_to_scale = X_test_raw.iloc[:, :num_cols_to_standardize]
        X_test_not_scaled = X_test_raw.iloc[:, num_cols_to_standardize:]
        scaler = StandardScaler()
        X_train_scaled_part = scaler.fit_transform(X_train_to_scale)
        X_test_scaled_part = scaler.transform(X_test_to_scale)
        if isinstance(X_train_not_scaled, pd.DataFrame):
             X_train_scaled = pd.concat([pd.DataFrame(X_train_scaled_part, columns=X_train_to_scale.columns, index=X_train_to_scale.index),
                                      X_train_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
             X_test_scaled = pd.concat([pd.DataFrame(X_test_scaled_part, columns=X_test_to_scale.columns, index=X_test_to_scale.index),
                                     X_test_not_scaled.reset_index(drop=True)], axis=1).to_numpy()
        else:
            X_train_scaled = np.hstack((X_train_scaled_part, X_train_not_scaled.to_numpy() if hasattr(X_train_not_scaled, 'to_numpy') else X_train_not_scaled))
            X_test_scaled = np.hstack((X_test_scaled_part, X_test_not_scaled.to_numpy() if hasattr(X_test_not_scaled, 'to_numpy') else X_test_not_scaled))
    else:
        print("不进行特征标准化处理。")
        X_train_scaled = X_train_raw.to_numpy()
        X_test_scaled = X_test_raw.to_numpy()
    print(f"数据加载和预处理完成。训练特征形状: {X_train_scaled.shape}, 测试特征形状: {X_test_scaled.shape}")
    return X_train_scaled, y_train, X_test_scaled, y_test, feature_names
# Cell 3: 训练和评估SVM (模板 - 为每个数据集修改参数)

# --- 手动设置参数 ---
class Args:
    pass

# === 定义所有要测试的数据集 (与之前的定义相同) ===
datasets_to_process = {
    "mimic3_36_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic4_36_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic3_eICU_8_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "mimic4_8_factors": {"standardize_all": True, "num_cols_to_standardize": None},
    "local_8_factors": {"standardize_all": True, "num_cols_to_standardize": None}
}
# !!! 重要: 对于36因子数据，如果原始 stander_data 只标准化了部分列，请修改上面的配置。
# =============================================================================

# --- 主循环，遍历所有数据集 ---
for dataset_tag, std_config in datasets_to_process.items():
    print(f"\n\n{'='*30} PROCESSING DATASET (SVM): {dataset_tag} {'='*30}")
    args = Args()

    # 1. 指定预分割数据所在的目录
    args.presplit_data_dir = f'./temp_data_utils_logs/{dataset_tag}_split'

    # 2. 指定输出目录
    args.output_dir = f'./traditional_ml_outputs/SVM_{dataset_tag}_notebook' # SVM for Support Vector Machine

    # 3. 标签列名
    args.label_column = "dead"

    # --- SVM 参数 (基于你的原始代码) ---
    # 你的原始 main() 中设置了 kernel = 'poly' 和 C = 1
    args.svm_C = 1.0
    args.svm_kernel = 'poly' # 可选: 'linear', 'poly', 'rbf', 'sigmoid'
    args.svm_degree = 3      # 当 kernel='poly'时使用，与原始代码一致
    args.svm_gamma = 'scale' # 'scale' (1 / (n_features * X.var())) or 'auto' (1 / n_features) or float
                             # 对于'poly', 'rbf', 'sigmoid'核。'scale'是较新的推荐默认值。
    args.svm_probability = True # 设置为True以便SVC直接输出概率，虽然CalibratedCV也会处理
    # 你可以根据需要为不同数据集或实验修改这些值
    # -----------------------------------------

    # --- 标准化选项 ---
    standardize_all_features_flag = std_config["standardize_all"]
    num_cols_to_standardize_value = std_config["num_cols_to_standardize"]
    # --------------------


    # --- 开始执行当前数据集的训练和评估逻辑 ---
    os.makedirs(args.output_dir, exist_ok=True)
    print(f"输出将保存到: {args.output_dir}")
    # print(f"参数: {vars(args)}")

    # 1. 加载和预处理数据
    print(f"\n--- 加载数据 for {dataset_tag} ---")
    try:
        X_train, y_train, X_test, y_test, feature_names = load_and_preprocess_data(
            presplit_dir=args.presplit_data_dir,
            label_col=args.label_column,
            train_file="split_train_data_seed256.csv",
            test_file="split_test_data_seed256.csv",
            standardize_all_features=standardize_all_features_flag,
            num_cols_to_standardize=num_cols_to_standardize_value
        )
    except FileNotFoundError as e:
        print(f"错误: 数据文件未找到 for {dataset_tag} at {args.presplit_data_dir}. 跳过此数据集。错误信息: {e}")
        continue
    except ValueError as e:
        print(f"错误: 加载数据时发生错误 for {dataset_tag}. 跳过此数据集。错误信息: {e}")
        continue


    # 2. 训练 SVM 模型
    print(f"\n--- 开始训练 SVM for {dataset_tag} ---")
    print(f"  C: {args.svm_C}, kernel: {args.svm_kernel}, degree: {args.svm_degree if args.svm_kernel=='poly' else 'N/A'}, gamma: {args.svm_gamma if args.svm_kernel in ['rbf', 'poly', 'sigmoid'] else 'N/A'}")
    start_time = time.time()

    # SVM对特征缩放非常敏感，确保数据已标准化
    base_svm_model = sklearn.svm.SVC(
        C=args.svm_C,
        kernel=args.svm_kernel,
        degree=args.svm_degree,
        gamma=args.svm_gamma,
        probability=args.svm_probability, # 设置为True以启用predict_proba
        random_state=256 # 为了可复现性
    )

    # 使用 CalibratedClassifierCV 进行概率校准
    # 对于SVC，如果 probability=True，它已经可以输出概率了，
    # 但CalibratedCV仍然可以用于进一步改善概率校准，特别是当cv的折数大于1时。
    # 如果SVC的probability=False(默认)，则CalibratedCV是必须的以获得概率。
    # 为了与你的原始流程一致，我们继续使用CalibratedCV。
    calibrated_model = sklearn.calibration.CalibratedClassifierCV(base_svm_model, method='isotonic', cv=5) # cv=5 是一个常用值
    
    print("开始拟合校准模型...")
    calibrated_model.fit(X_train, y_train)
    print("模型拟合完成。")


    model_save_path = os.path.join(args.output_dir, "svm_calibrated.joblib")
    joblib.dump(calibrated_model, model_save_path)
    print(f"模型训练完成并在 {time.time() - start_time:.3f}s 内保存至 {model_save_path}")

    # 3. 在测试集上评估并获取所需输出
    y_pred_probs_test = calibrated_model.predict_proba(X_test)
    y_pred_class_test = calibrated_model.predict(X_test)
    positive_class_probs_test = y_pred_probs_test[:, 1]

    test_acc = accuracy_score(y_test, y_pred_class_test)
    if len(np.unique(y_test)) > 1:
        test_auc = roc_auc_score(y_test, positive_class_probs_test)
    else:
        print(f"警告: 数据集 {dataset_tag} 的测试集只包含一个类别，AUC无法计算，设为0.0。")
        test_auc = 0.0
    test_f1 = f1_score(y_test, y_pred_class_test, average='binary', zero_division=0)
    test_mcc = matthews_corrcoef(y_test, y_pred_class_test)
    test_cm = confusion_matrix(y_test, y_pred_class_test)

    print("\n--- 测试集评估结果 ---")
    print(f"  准确率 (Accuracy): {test_acc:.4f}")
    print(f"  AUC: {test_auc:.4f}")
    print(f"  F1 分数: {test_f1:.4f}")
    print(f"  MCC: {test_mcc:.4f}")
    print(f"  混淆矩阵:\n{test_cm}")

    # 4. 保存DeLong检验所需的文件
    np.save(os.path.join(args.output_dir, "test_labels.npy"), y_test)
    np.save(os.path.join(args.output_dir, "test_probabilities.npy"), positive_class_probs_test)
    print(f"\n测试集真实标签和预测概率已保存至: {args.output_dir}")

    # 5. 保存指标摘要
    tn_val, fp_val, fn_val, tp_val = (0,0,0,0)
    if test_cm.size == 4: tn_val, fp_val, fn_val, tp_val = test_cm.ravel()
    elif test_cm.size == 1 and len(y_test) > 0:
        if y_test[0] == 0 and y_pred_class_test[0] == 0 : tn_val = test_cm[0,0]
        elif y_test[0] == 1 and y_pred_class_test[0] == 1 : tp_val = test_cm[0,0]

    metrics_summary = {
        "model_name": "SVM", # 修改模型名称
        "dataset_tag": dataset_tag,
        "presplit_data_dir": args.presplit_data_dir,
        "svm_C": args.svm_C,
        "svm_kernel": args.svm_kernel,
        "svm_degree": args.svm_degree if args.svm_kernel=='poly' else "N/A",
        "svm_gamma": args.svm_gamma if args.svm_kernel in ['rbf', 'poly', 'sigmoid'] else "N/A",
        "test_accuracy": test_acc,
        "test_auc": test_auc,
        "test_f1": test_f1,
        "test_mcc": test_mcc,
        "confusion_matrix_tn": tn_val,
        "confusion_matrix_fp": fp_val,
        "confusion_matrix_fn": fn_val,
        "confusion_matrix_tp": tp_val,
    }
    metrics_df = pd.DataFrame([metrics_summary])
    metrics_df.to_csv(os.path.join(args.output_dir, "metrics_summary.csv"), index=False)
    print(f"指标摘要已保存至: {os.path.join(args.output_dir, 'metrics_summary.csv')}")

    print(f"\nSVM模型 ({dataset_tag}) 训练和评估完成。")

print(f"\n\n{'='*30} 所有数据集处理完毕 (SVM) {'='*30}")




============================== PROCESSING DATASET (SVM): mimic3_36_factors ==============================
输出将保存到: ./traditional_ml_outputs/SVM_mimic3_36_factors_notebook

--- 加载数据 for mimic3_36_factors ---
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
对所有特征列进行标准化 (基于训练集拟合)...
数据加载和预处理完成。训练特征形状: (2667, 36), 测试特征形状: (1144, 36)

--- 开始训练 SVM for mimic3_36_factors ---
  C: 1.0, kernel: poly, degree: 3, gamma: scale
开始拟合校准模型...
模型拟合完成。
模型训练完成并在 3.731s 内保存至 ./traditional_ml_outputs/SVM_mimic3_36_factors_notebook/svm_calibrated.joblib

--- 测试集评估结果 ---
  准确率 (Accuracy): 0.7850
  AUC: 0.8429
  F1 分数: 0.7984
  MCC: 0.5721
  混淆矩阵:
[[411 152]
 [ 94 487]]

测试集真实标签和预测概率已保存至: ./traditional_ml_outputs/SVM_mimic3_36_factors_notebook
指标摘要已保存至: ./traditional_ml_outputs/SVM_mimic3_36_factors_notebook/metrics_summary.csv

SVM模型 (mimic3_36_factors) 训练和评估完成。


===============